In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DB_PATH, FIGURES_DIR, ensure_dirs

ensure_dirs()

from src.viz import apply_chinese, get_font_prop

font_prop = get_font_prop()
plt.rcParams["axes.unicode_minus"] = False

con = duckdb.connect(str(DB_PATH), read_only=True)


In [ ]:
rfm = con.sql("""
WITH user_stats AS (
SELECT
    user_id,
    COUNT(*) AS frequency,
    SUM(IF(behavior_type = '购买', 1,0)) AS buy_cnt
FROM user_behavior
GROUP BY user_id
)
SELECT
    CASE
        WHEN buy_cnt >= 5 AND frequency >= 50 THEN '高价值'
        WHEN buy_cnt >= 1 AND frequency >= 20 THEN '活跃'
        WHEN buy_cnt >= 0 AND frequency >= 10 THEN '浏览型'
        ELSE '低活跃'
    END AS user_level,
    COUNT(*) AS user_cnt
FROM user_stats
GROUP BY user_level
""").df()
print(rfm)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    rfm["user_cnt"],
    labels=rfm["user_level"],
    autopct="%1.1f%%",
    colors=["#4C72B0", "#55A868", "#DD8452", "#C44E52"],
    startangle=90,
    textprops={"fontproperties": font_prop},
)
ax.set_title("用户分层占比")
apply_chinese(ax, font_prop)
plt.savefig(FIGURES_DIR / "rfm_pie.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=rfm,
    x="user_level",
    y="user_cnt",
    ax=ax,
    hue="user_level",
    palette="Set2",
    legend=False,
)
for i, v in enumerate(rfm["user_cnt"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_title("各层级用户数", fontproperties = font_prop)
ax.set_xlabel("用户层级", fontproperties = font_prop)
ax.set_ylabel("用户数", fontproperties=font_prop)
for label in ax.get_xticklabels():
    label.set_fontproperties(font_prop)
apply_chinese(ax, font_prop)
plt.savefig(FIGURES_DIR/"rfm_bar.png")
plt.show()